In [1]:
# Environment Setup
import os
import warnings
warnings.filterwarnings('ignore')

from dotenv import load_dotenv
load_dotenv()

# Verify API key
if os.getenv("GEMINI_API_KEY"):
    print("✅ GEMINI_API_KEY found")
else:
    print("❌ GEMINI_API_KEY not found - please set it in your .env file")

✅ GEMINI_API_KEY found


In [2]:
# Core Imports

# Standard library
import numpy as np
import pandas as pd
import asyncio
import json

# LangChain components
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

# RAGAS components
from ragas import SingleTurnSample, EvaluationDataset, evaluate
from ragas.metrics import (
    Faithfulness,
    ResponseRelevancy,
    LLMContextPrecisionWithReference,
    LLMContextRecall,
    ContextEntityRecall,
    NoiseSensitivity
)
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

print("✅ All imports successful")

✅ All imports successful


In [3]:
# Initialize base models (GEMINI)
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

# Wrap for RAGAS compatibility
ragas_llm = LangchainLLMWrapper(llm)
ragas_embeddings = LangchainEmbeddingsWrapper(embeddings)

print("✅ LLM initialized:gemini-2.5-flash")
print("✅ Embeddings initialized: models/gemini-embedding-001")
print("✅ RAGAS wrappers ready")

✅ LLM initialized:gemini-2.5-flash
✅ Embeddings initialized: models/gemini-embedding-001
✅ RAGAS wrappers ready


In [4]:
# Helper function for running async code in Jupyter

def run_async(coro):
    """Helper to run async code in Jupyter notebooks"""
    try:
        loop = asyncio.get_event_loop()
        if loop.is_running():
            # We're in Jupyter with an existing loop
            import nest_asyncio
            nest_asyncio.apply()
            return loop.run_until_complete(coro)
        else:
            return asyncio.run(coro)
    except RuntimeError:
        return asyncio.run(coro)

print("✅ Async helper ready")

✅ Async helper ready


In [5]:
# Define our test case

# The response we want to evaluate
test_response = "The first Super Bowl was held on January 15, 1967 in Los Angeles. It was a sunny day with clear skies."

# The context that was retrieved (source of truth)
test_context = [
    "The First AFL-NFL World Championship Game was played on December 20, 2024, at the Los Angeles Memorial Coliseum in Newyork."
]

print("📝 Response to evaluate:")
print(f"   '{test_response}'")
print("\n📚 Retrieved context:")
print(f"   '{test_context[0]}'")

📝 Response to evaluate:
   'The first Super Bowl was held on January 15, 1967 in Los Angeles. It was a sunny day with clear skies.'

📚 Retrieved context:
   'The First AFL-NFL World Championship Game was played on December 20, 2024, at the Los Angeles Memorial Coliseum in Newyork.'


In [6]:
# Run actual RAGAS Faithfulness metric

# Create sample in RAGAS format
faithfulness_sample = SingleTurnSample(
    user_input="When was the first Super Bowl?",
    response=test_response,
    retrieved_contexts=test_context
)

# Initialize and run the metric
faithfulness_metric = Faithfulness(llm=ragas_llm)

ragas_faithfulness = run_async(faithfulness_metric.single_turn_ascore(faithfulness_sample))

print("🔬 RAGAS Faithfulness Result")
print("=" * 50)
print(f"   RAGAS metric score:  {ragas_faithfulness:.2f}")

🔬 RAGAS Faithfulness Result
   RAGAS metric score:  0.00


In [7]:
# Compare different faithfulness scenarios

faithfulness_examples = [
    {
        "name": "Perfect Faithfulness (No hallucinations)",
        "response": "The first Super Bowl was played on January 15, 1967 at the Los Angeles Memorial Coliseum.",
        "context": ["The First AFL-NFL World Championship Game was played on January 15, 1967, at the Los Angeles Memorial Coliseum."]
    },
    {
        "name": "Partial Faithfulness (Some hallucinations)",
        "response": "The first Super Bowl was on January 15, 1967. The Green Bay Packers won 35-10 with Bart Starr as MVP.",
        "context": ["The First AFL-NFL World Championship Game was played on January 15, 1967."]
    },
    {
        "name": "Zero Faithfulness (Complete hallucination)",
        "response": "The first Super Bowl was held in Miami in 1970 and attracted over 100,000 spectators.",
        "context": ["The First AFL-NFL World Championship Game was played on January 15, 1967, at the Los Angeles Memorial Coliseum."]
    }
]

print("📊 Faithfulness Comparison: Different Scenarios")
print("=" * 70)

for example in faithfulness_examples:
    sample = SingleTurnSample(
        user_input="Tell me about the first Super Bowl",
        response=example["response"],
        retrieved_contexts=example["context"]
    )
    score = run_async(faithfulness_metric.single_turn_ascore(sample))
    
    print(f"\n🏷️  {example['name']}")
    print(f"   Response: '{example['response'][:80]}...'" if len(example['response']) > 80 else f"   Response: '{example['response']}'")
    print(f"   Score: {score:.2f}")

📊 Faithfulness Comparison: Different Scenarios

🏷️  Perfect Faithfulness (No hallucinations)
   Response: 'The first Super Bowl was played on January 15, 1967 at the Los Angeles Memorial ...'
   Score: 1.00

🏷️  Partial Faithfulness (Some hallucinations)
   Response: 'The first Super Bowl was on January 15, 1967. The Green Bay Packers won 35-10 wi...'
   Score: 0.25

🏷️  Zero Faithfulness (Complete hallucination)
   Response: 'The first Super Bowl was held in Miami in 1970 and attracted over 100,000 specta...'
   Score: 0.00


Answer Relevancy

Answer Relevancy checks if the answer actually answers the question asked. It doesn't care if the answer is factually correct - just whether it's relevant to the question.

In [8]:
# Define our test case for relevancy

original_question = "When was the first Super Bowl?"
test_answer = "The first Super Bowl was held on January 15, 1967"

print("📝 Original Question:")
print(f"   '{original_question}'")
print("\n📝 Answer to Evaluate:")
print(f"   '{test_answer}'")

📝 Original Question:
   'When was the first Super Bowl?'

📝 Answer to Evaluate:
   'The first Super Bowl was held on January 15, 1967'


In [10]:
# Run actual RAGAS Answer Relevancy metric

relevancy_sample = SingleTurnSample(
    user_input=original_question,
    response=test_answer,
    retrieved_contexts=["The First AFL-NFL World Championship Game was played on January 15, 1967."]
)

relevancy_metric = ResponseRelevancy(llm=ragas_llm, embeddings=ragas_embeddings)

ragas_relevancy = run_async(relevancy_metric.single_turn_ascore(relevancy_sample))

print("🔬 RAGAS Answer Relevancy Result")
print("=" * 50)
print(f"   RAGAS metric score:  {ragas_relevancy:.4f}")


🔬 RAGAS Answer Relevancy Result
   RAGAS metric score:  0.7929


In [11]:
# Compare different relevancy scenarios

relevancy_examples = [
    {
        "name": "Highly Relevant (Directly answers WHEN)",
        "question": "When was the first Super Bowl?",
        "answer": "The first Super Bowl was held on January 15, 1967.",
    },
    {
        "name": "Partially Relevant (Answers but adds extra info)",
        "question": "When was the first Super Bowl?",
        "answer": "The Super Bowl is the annual championship game of the NFL, first held on January 15, 1967.",
    },
    {
        "name": "Low Relevancy (Doesn't answer WHEN)",
        "question": "When was the first Super Bowl?",
        "answer": "The Super Bowl is the annual championship game of the National Football League.",
    },
    {
        "name": "Off-topic (Completely irrelevant)",
        "question": "When was the first Super Bowl?",
        "answer": "Pizza is a popular Italian dish that spread worldwide in the 20th century.",
    }
]

print("📊 Answer Relevancy Comparison")
print("=" * 70)

for example in relevancy_examples:
    sample = SingleTurnSample(
        user_input=example["question"],
        response=example["answer"],
        retrieved_contexts=["Context not relevant for this metric."]
    )
    score = run_async(relevancy_metric.single_turn_ascore(sample))
    
    print(f"\n🏷️  {example['name']}")
    print(f"   Q: '{example['question']}'")
    print(f"   A: '{example['answer'][:60]}...'" if len(example['answer']) > 60 else f"   A: '{example['answer']}'")
    print(f"   Score: {score:.4f}")

📊 Answer Relevancy Comparison

🏷️  Highly Relevant (Directly answers WHEN)
   Q: 'When was the first Super Bowl?'
   A: 'The first Super Bowl was held on January 15, 1967.'
   Score: 0.7929

🏷️  Partially Relevant (Answers but adds extra info)
   Q: 'When was the first Super Bowl?'
   A: 'The Super Bowl is the annual championship game of the NFL, f...'
   Score: 0.7586

🏷️  Low Relevancy (Doesn't answer WHEN)
   Q: 'When was the first Super Bowl?'
   A: 'The Super Bowl is the annual championship game of the Nation...'
   Score: 0.7170

🏷️  Off-topic (Completely irrelevant)
   Q: 'When was the first Super Bowl?'
   A: 'Pizza is a popular Italian dish that spread worldwide in the...'
   Score: 0.5612
